In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from enum import Enum
from dataclasses import dataclass
import math
from tqdm import tqdm
import sys, inspect

from tiles import *
from utils import *
import solver as solver_import

In [ ]:
def plot_schedule(schedule):
    methods = inspect.getmembers(schedule, inspect.ismethod)
    methods = [(i,j) for i,j in methods if not i.startswith('_')]

    com_ax = plt.subplot()
    com_ax.set_title('Combined')

    fig, ax = plt.subplots(1, len(methods), figsize=(3.5*len(methods), 3))

    for i, (m_name, m_func) in enumerate(methods):
        # print(i, m_name,m_func)
        g = list(map(m_func, np.arange(0,schedule.max_iter))) 
        ax[i].plot(g)
        ax[i].set_title(f'{m_name} max:{max(g):.3f} min:{min(g):.3f}')

        com_ax.plot(g)
    plt.plot()

In [ ]:
class Schedule:
    def __init__(self, max_iter):
        self.max_iter = max_iter
    def lr(self,x):
        x /= self.max_iter
        return ( np.cos(x*2*np.pi/4)**32 * 5 + np.cos(x*2*np.pi/4)**2 + np.cos(x*2*np.pi*5 % (np.pi/2))/5 + np.cos(x*2*np.pi*2 % (np.pi/2))/2 ) /6.7

    def temp(self,x):
        x /= self.max_iter
        if x<0.5:
            return np.maximum( np.cos(x*2*np.pi/2)**2 + np.cos(x*2*np.pi*5 % (np.pi/2))/5 , 0.001) / 1.2
        else:
            return max(0.2*(1-(x-0.5)*4),0)
    def logits_penalty(self,x):
        x /= self.max_iter
        return np.cos(x*np.pi/2)**2 
    
    def decision_penalty(self,x):
        x /= self.max_iter

        x /=2
        return np.cos(x*np.pi/2 + 1/2*np.pi)**4 *4
    def empty_pos_penalty(self,x):
        x /= self.max_iter

        x /=2
        return np.cos(x*np.pi/2 + 1/2*np.pi)**8 *4 / 0.24

schedule = Schedule(150)
plot_schedule(schedule)

In [ ]:
BOX_SIZE = 15
INPUT_POS = (0,3)
OUT_POS = (13, 12)
NEG_INF = -500.0

In [ ]:
import importlib
importlib.reload(solver_import) # Fast reaload

sim = solver_import.FactorioSim(BOX_SIZE, DEFAULT_TILE_DATA)

In [ ]:
logits = torch.full((BOX_SIZE,BOX_SIZE,len(Tile)), -500.0)
active_tiles = list(range(0,12+1))

logits[:,:,active_tiles] = 5 * torch.randn(BOX_SIZE,BOX_SIZE,len(active_tiles))

logits.requires_grad_()
# solve = smax(solve)

add_tensor = torch.zeros(BOX_SIZE,BOX_SIZE)
add_tensor[INPUT_POS[0], INPUT_POS[1]] = 100

MAX_ITER = 151
START_TEMP = 100

opt = torch.optim.Adam([logits],lr=100)
# opt = torch.optim.SGD([logits], lr=20,momentum=0.98,nesterov=True)
# opt = torch.optim.RMSprop([logits],lr=5, momentum=0.98)
# sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=MAX_ITER)
for iter in range(MAX_ITER):
    for g in opt.param_groups:
        g['lr'] = 100 * schedule.lr(iter)

    logits_noise = logits + 0.01 * schedule.temp(iter) * torch.randn_like(logits)
    with torch.no_grad():
        logits_noise[:, :, 12::] = -200
    solve = torch.nn.functional.log_softmax(logits_noise/50,dim=-1)
    # solve = gumbel_softmax_log(logits_noise,temperature=5)

    if iter%10==0:
        plot_solve(solve)
    items = sim.eval(solve,add_tensor, max_iter=BOX_SIZE*1.8)
    # path_loss = -items[0, 13, :].mean(dim=-1)
    path_loss = -( items[OUT_POS[0], OUT_POS[1], :].max(dim=-1)[0] - np.log(100) )

    inserter_penalty = (logits_noise[:,:,[i.value for i in Tile.INSERTERS()]]**2).mean() * 100 # * 3.5 # 
    inserter_penalty1 = (logits_noise[:,:,[i.value for i in Tile.INSERTERS()]]).mean() * 100 # * 3.5 # 
    # belt_penalty = (logits_noise[:,:,[i.value for i in Tile.BELTS()]]**2).mean() * 1 *0.001
    belt_penalty = (torch.sigmoid(solve[:,:,[i.value for i in Tile.BELTS()]])).mean() * 1 * 10
    # belt_penalty = (torch.sigmoid(solve[:,:,[i.value for i in Tile.BELTS()]])).mean() * 1 * 10

    # inserter_penalty += (solve[:,:,[i.value for i in Tile.INSERTERS()]]).exp().mean() * 7000 * 10 # * 3.5 #  # 
    # belt_penalty += (solve[:,:,[i.value for i in Tile.BELTS()]]).exp().mean() * 1

    logits_penalty = (torch.clamp(logits_noise**2,min=200.0)-200.0).mean() * 0.01 * schedule.logits_penalty(iter)
    logits_penalty1 = (solve[:,:,1::]).mean() * 10#  * schedule.logits_penalty(iter)
    
    empty_pos_penalty = -(torch.clamp(logits_noise[:,:,0],max=400)).mean() * 1 /4 * schedule.empty_pos_penalty(iter)
    dec_penalty = (torch.minimum(1-solve,solve-1)**2).mean() * 0.3 * schedule.decision_penalty(iter)
    
    loss = path_loss + inserter_penalty1 + belt_penalty + dec_penalty +logits_penalty + empty_pos_penalty + logits_penalty1# + + dec_penalty 
    loss.backward()
    opt.step()

    grad_norm = logits.grad.abs().mean().item()
    empty_ratio = (solve[:, :, 0] > -0.5).float().mean().item()
    # print(loss.item(), path_loss.item(),belt_penalty.item(),inserter_penalty.item(),temp, grad_norm, empty_ratio, sch.get_last_lr()[0])
    print(f'{iter/MAX_ITER*100:.1f}% Loss comb:{loss.item():.3f}, Path loss:{path_loss:.3f}, Belt penalty:{belt_penalty:.2f}, Inser penalty:{inserter_penalty:.2f}, Logits penalty: {logits_penalty:.2f}, Logits penalty1: {logits_penalty:.2f}, Decision penalty: {dec_penalty:.2f} Empty:{empty_pos_penalty:.2f}')

    opt.zero_grad()
    # sch.step()

plot_solve(solve)

In [ ]:
sim.eval(solve, iter_add=add_tensor, verbose=10,max_iter=20)

In [ ]:
render_solve(solve)